In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("catalog_name", "retail_lakehouse")
dbutils.widgets.text("pipeline_run_id", "")

In [0]:
import uuid

environment = dbutils.widgets.get("environment").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

In [0]:
#Reading Gold Facts and Dimensions

from pyspark.sql import functions as F

fact_sales_df = spark.table(
    f"{catalog_name}.gold.fact_sales"
)

fact_inventory_df = spark.table(
    f"{catalog_name}.gold.fact_inventory"
)

dim_product_df = spark.table(
    f"{catalog_name}.gold.dim_product"
)

dim_store_df = spark.table(
    f"{catalog_name}.gold.dim_store"
)

dim_date_df = spark.table(
    f"{catalog_name}.gold.dim_date"
)

In [0]:
#Daily store-sales KPI

daily_store_sales_df = (
    fact_sales_df
    .groupBy(
        "date_key",
        "store_key"
    )
    .agg(
        F.countDistinct("transaction_id")
            .alias("total_transactions"),
        F.sum("quantity")
            .alias("units_sold"),
        F.sum("gross_amount")
            .alias("gross_sales_amount"),
        F.sum("discount_amount")
            .alias("total_discount_amount"),
        F.sum("recognized_sales_amount")
            .alias("net_recognized_sales"),
        F.sum("gross_profit_amount")
            .alias("gross_profit_amount"),
        F.sum("return_flag")
            .alias("returned_transactions"),
        F.sum("cancellation_flag")
            .alias("cancelled_transactions"),
        F.avg("net_amount")
            .alias("average_transaction_value")
    )
    .withColumn(
        "return_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("returned_transactions")
            / F.col("total_transactions")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "cancellation_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.col("cancelled_transactions")
            / F.col("total_transactions")
        ).otherwise(F.lit(0.0))
    )
)

(
    daily_store_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog_name}.gold.daily_store_sales"
    )
)

In [0]:
#Monthly product performance KPI

monthly_product_performance_df = (
    fact_sales_df.alias("sales")
    .join(
        dim_date_df.select(
            "date_key",
            "calendar_year",
            "calendar_month",
            "year_month"
        ),
        on="date_key",
        how="inner"
    )
    .groupBy(
        "year_month",
        "calendar_year",
        "calendar_month",
        "product_key"
    )
    .agg(
        F.countDistinct("transaction_id")
            .alias("transaction_count"),
        F.sum("quantity")
            .alias("units_sold"),
        F.sum("recognized_sales_amount")
            .alias("recognized_sales_amount"),
        F.sum("gross_profit_amount")
            .alias("gross_profit_amount"),
        F.sum("return_flag")
            .alias("return_count")
    )
    .withColumn(
        "product_return_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("return_count")
            / F.col("transaction_count")
        ).otherwise(F.lit(0.0))
    )
)

(
    monthly_product_performance_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog_name}.gold.monthly_product_performance"
    )
)

In [0]:
#Inventory performance KPI
#Average inventory = (opening stock + closing stock) / 2
#Inventory turnover = units sold / average inventory

inventory_performance_df = (
    fact_inventory_df
    .withColumn(
        "average_inventory",
        (
            F.col("opening_stock")
            + F.col("closing_stock")
        ) / F.lit(2.0)
    )
    .withColumn(
        "inventory_turnover_ratio",
        F.when(
            F.col("average_inventory") > 0,
            F.col("sold_quantity")
            / F.col("average_inventory")
        ).otherwise(F.lit(0.0))
    )
)

(
    inventory_performance_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog_name}.gold.inventory_performance"
    )
)

In [0]:
#Store-sales reporting view

spark.sql(f"""
CREATE OR REPLACE VIEW
{catalog_name}.gold.vw_store_sales_summary
AS
SELECT
    d.full_date,
    d.calendar_year,
    d.calendar_month,
    d.month_name,
    d.year_month,
    s.store_id,
    s.store_name,
    s.city,
    s.state,
    s.region,
    s.store_size,
    k.total_transactions,
    k.units_sold,
    k.gross_sales_amount,
    k.total_discount_amount,
    k.net_recognized_sales,
    k.gross_profit_amount,
    k.returned_transactions,
    k.cancelled_transactions,
    k.return_rate,
    k.cancellation_rate,
    k.average_transaction_value
FROM {catalog_name}.gold.daily_store_sales k
JOIN {catalog_name}.gold.dim_date d
    ON k.date_key = d.date_key
JOIN {catalog_name}.gold.dim_store s
    ON k.store_key = s.store_key
""")

In [0]:
#Product performance view

spark.sql(f"""
CREATE OR REPLACE VIEW
{catalog_name}.gold.vw_product_performance
AS
SELECT
    p.year_month,
    p.calendar_year,
    p.calendar_month,
    d.product_id,
    d.product_name,
    d.category,
    d.brand,
    p.transaction_count,
    p.units_sold,
    p.recognized_sales_amount,
    p.gross_profit_amount,
    p.return_count,
    p.product_return_rate
FROM {catalog_name}.gold.monthly_product_performance p
JOIN {catalog_name}.gold.dim_product d
    ON p.product_key = d.product_key
""")



In [0]:
#Creating Inventory view

spark.sql(f"""
CREATE OR REPLACE VIEW
{catalog_name}.gold.vw_inventory_performance
AS
SELECT
    i.inventory_date,
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    s.store_id,
    s.store_name,
    s.region,
    i.opening_stock,
    i.received_quantity,
    i.sold_quantity,
    i.damaged_quantity,
    i.closing_stock,
    i.stock_out_flag,
    i.low_stock_flag,
    i.average_inventory,
    i.inventory_turnover_ratio
FROM {catalog_name}.gold.inventory_performance i
JOIN {catalog_name}.gold.dim_product p
    ON i.product_key = p.product_key
JOIN {catalog_name}.gold.dim_store s
    ON i.store_key = s.store_key
""")

In [0]:
#Validating the views

display(
    spark.table(
        f"{catalog_name}.gold.vw_store_sales_summary"
    ).limit(20)
)

display(
    spark.table(
        f"{catalog_name}.gold.vw_product_performance"
    ).limit(20)
)

display(
    spark.table(
        f"{catalog_name}.gold.vw_inventory_performance"
    ).limit(20)
)